# Prompt Engineering Project: Controlled Content Generation

**Learning & Communications Team — Internal Microlearning Newsletter**

This notebook walks through a full prompt development cycle (design → generate → evaluate → refine) to transform a dense internal security policy paragraph into clear, employee-facing microlearning content.

**Concepts covered:** tone / structure / format / length control, hallucination detection & mitigation, paraphrasing vs. quoting.

**Techniques:** role prompting, instruction-based prompting, output constraints, iterative evaluation & revision.

---

## Source Material (Input Text)

> Employees must ensure that all remote access to internal systems is established via the approved secure VPN. Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used to access proprietary data or sensitive communications.


In [1]:
ORIGINAL_TEXT = """Employees must ensure that all remote access to internal systems is established via the approved secure VPN. Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used to access proprietary data or sensitive communications."""

print(ORIGINAL_TEXT)
print("\nWord count:", len(ORIGINAL_TEXT.split()))

Employees must ensure that all remote access to internal systems is established via the approved secure VPN. Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used to access proprietary data or sensitive communications.

Word count: 38


---
## Step 1 — Craft a Prompt That Controls Tone, Format, and Length

**Goal:** rewrite in a friendly, clear tone, using bullet points, paraphrased (no direct quotes), under 75 words.

The prompt below combines **role prompting** ("You are a friendly internal communications writer"), **instruction-based prompting** (explicit task), and **output constraints** (tone, format, length, paraphrase).

In [2]:
PROMPT_V1 = """You are a friendly internal communications writer creating a microlearning snippet for a company newsletter.

Rewrite the policy text below for a general employee audience.

Requirements:
- Use a warm, friendly, and clear tone.
- Organize the information as bullet points.
- Paraphrase the content in your own words; do NOT quote the original directly.
- Keep the entire output under 75 words.

Policy text:
\"\"\"
{text}
\"\"\"
""".format(text=ORIGINAL_TEXT)

print(PROMPT_V1)

You are a friendly internal communications writer creating a microlearning snippet for a company newsletter.

Rewrite the policy text below for a general employee audience.

Requirements:
- Use a warm, friendly, and clear tone.
- Organize the information as bullet points.
- Paraphrase the content in your own words; do NOT quote the original directly.
- Keep the entire output under 75 words.

Policy text:
"""
Employees must ensure that all remote access to internal systems is established via the approved secure VPN. Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used to access proprietary data or sensitive communications.
"""



### Sample Generated Output (Step 1)

*(Output produced by the generative model when given PROMPT_V1.)*

**Staying Secure When Working Remotely 🔐**

- Always connect through the company's approved secure VPN whenever you log in from outside the office.
- Avoid using open or unsecured networks to reach internal systems.
- Don't use personal devices that don't have proper security protection installed.
- Keep proprietary data and private conversations safe by following these simple steps.

*Word count: ~58 words*

---
## Step 2 — Evaluate the Output

We score the Step 1 output against the rubric.

In [3]:
sample_output_v1 = """Staying Secure When Working Remotely
- Always connect through the company's approved secure VPN whenever you log in from outside the office.
- Avoid using open or unsecured networks to reach internal systems.
- Don't use personal devices that don't have proper security protection installed.
- Keep proprietary data and private conversations safe by following these simple steps."""

print("Word count:", len(sample_output_v1.split()))

evaluation = {
    "Relevance":        ("Pass", "Covers VPN, no unsecured connections, no unprotected personal devices, protecting sensitive data."),
    "Clarity":          ("Pass", "Plain language; understandable by non-technical staff."),
    "Structure":        ("Pass", "Formatted as clear bullet points."),
    "Tone":             ("Pass", "Warm and friendly, appropriate for internal comms."),
    "Length":           ("Pass", "Under 75 words (~58)."),
    "Factual Accuracy": ("Watch", "Faithful to source. Risk: model could add tech (MFA, antivirus brands) not in input."),
}

for k, (verdict, note) in evaluation.items():
    print(f"{k:18s} [{verdict:5s}] {note}")

Word count: 58
Relevance          [Pass ] Covers VPN, no unsecured connections, no unprotected personal devices, protecting sensitive data.
Clarity            [Pass ] Plain language; understandable by non-technical staff.
Structure          [Pass ] Formatted as clear bullet points.
Tone               [Pass ] Warm and friendly, appropriate for internal comms.
Length             [Pass ] Under 75 words (~58).
Factual Accuracy   [Watch] Faithful to source. Risk: model could add tech (MFA, antivirus brands) not in input.


---
## Step 3 — Detect and Mitigate Hallucinations

**Hallucination risk:** the model may invent policy details not in the source — e.g. naming specific VPN products, requiring multi-factor authentication, antivirus brands, or password rules. None of these appear in the input.

**Mitigation:** add a strict grounding constraint that limits the model to the input only.

In [4]:
PROMPT_V1_REVISED = """You are a friendly internal communications writer creating a microlearning snippet for a company newsletter.

Rewrite the policy text below for a general employee audience.

Requirements:
- Use a warm, friendly, and clear tone.
- Organize the information as bullet points.
- Paraphrase the content in your own words; do NOT quote the original directly.
- Keep the entire output under 75 words.

STRICT GROUNDING RULES:
- Only paraphrase the content in the input text.
- Do NOT add new recommendations, rules, technologies, product names, or examples.
- Do NOT remove any of the three core points: secure VPN, no unsecured connections, no personal devices lacking endpoint protection.

Policy text:
\"\"\"
{text}
\"\"\"
""".format(text=ORIGINAL_TEXT)

print(PROMPT_V1_REVISED)

You are a friendly internal communications writer creating a microlearning snippet for a company newsletter.

Rewrite the policy text below for a general employee audience.

Requirements:
- Use a warm, friendly, and clear tone.
- Organize the information as bullet points.
- Paraphrase the content in your own words; do NOT quote the original directly.
- Keep the entire output under 75 words.

STRICT GROUNDING RULES:
- Only paraphrase the content in the input text.
- Do NOT add new recommendations, rules, technologies, product names, or examples.
- Do NOT remove any of the three core points: secure VPN, no unsecured connections, no personal devices lacking endpoint protection.

Policy text:
"""
Employees must ensure that all remote access to internal systems is established via the approved secure VPN. Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used to access proprietary data or sensitive communications.
"""



### Sample Grounded Output (Step 3)

**Working Remotely? Keep It Secure 🔐**

- Always use the company's approved secure VPN to reach internal systems from afar.
- Never connect through unsecured networks.
- Don't use personal devices that lack endpoint protection.
- These steps keep proprietary data and sensitive communications safe.

*Word count: ~44 words — no invented technologies or extra rules.*

---
## Step 4 — Paraphrasing Deep Dive (Junior Intern Audience)

A second prompt variant aimed at junior interns: plain language, short phrases, max 4 bullets, no jargon, supportive tone.

In [5]:
PROMPT_INTERN = """You are a supportive mentor explaining a security rule to a brand-new junior intern.

Rewrite the policy text below in very plain, everyday language.

Requirements:
- Use a supportive, encouraging, and informative tone.
- Use short, simple phrases.
- Use no more than 4 bullet points.
- Avoid all corporate, legal, or technical jargon (explain ideas simply).
- Only use information found in the input text; do not add new rules or technologies.

Policy text:
\"\"\"
{text}
\"\"\"
""".format(text=ORIGINAL_TEXT)

print(PROMPT_INTERN)

You are a supportive mentor explaining a security rule to a brand-new junior intern.

Rewrite the policy text below in very plain, everyday language.

Requirements:
- Use a supportive, encouraging, and informative tone.
- Use short, simple phrases.
- Use no more than 4 bullet points.
- Avoid all corporate, legal, or technical jargon (explain ideas simply).
- Only use information found in the input text; do not add new rules or technologies.

Policy text:
"""
Employees must ensure that all remote access to internal systems is established via the approved secure VPN. Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used to access proprietary data or sensitive communications.
"""



### Sample Intern Output (Step 4)

**Welcome! Here's how to stay safe online at work 🙂**

- Log in through the company's secure VPN when you work from home.
- Skip public or open Wi-Fi for work — it isn't safe.
- Only use a device that has the right security set up on it.
- Doing this keeps our private info protected. You've got this!

*Plain language, 4 bullets, no jargon, supportive tone.*

---
## Step 5 — Quote Extraction Variant

Instead of paraphrasing, extract the single direct quote that best captures the core security policy.

In [6]:
PROMPT_QUOTE = """From the policy text below, extract ONE direct quote (word-for-word, unchanged) that best captures the core security requirement.

Requirements:
- Copy the sentence exactly as written, with no edits.
- Wrap it in quotation marks.
- Do not paraphrase, summarize, or add commentary.

Policy text:
\"\"\"
{text}
\"\"\"
""".format(text=ORIGINAL_TEXT)

print(PROMPT_QUOTE)

From the policy text below, extract ONE direct quote (word-for-word, unchanged) that best captures the core security requirement.

Requirements:
- Copy the sentence exactly as written, with no edits.
- Wrap it in quotation marks.
- Do not paraphrase, summarize, or add commentary.

Policy text:
"""
Employees must ensure that all remote access to internal systems is established via the approved secure VPN. Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used to access proprietary data or sensitive communications.
"""



### Extracted Quote (Step 5)

> "Employees must ensure that all remote access to internal systems is established via the approved secure VPN."

This single sentence carries the central, non-negotiable requirement of the policy.

---

### Reflection: Quoting vs. Paraphrasing

**When is quoting more appropriate than paraphrasing?**

- **Compliance, legal, and policy communications**, where the exact wording carries authority and must not be softened or reinterpreted (e.g., official policy bulletins, mandatory acknowledgements, audit documentation).
- **Disciplinary or contractual notices**, where staff are being held to a specific, literal standard.
- When you want to signal that the statement is the **organization's official position**, not the writer's interpretation — quoting preserves accountability and traceability to the source.

**When might quoting pose a risk?**

- **Reduced comprehension:** dense, jargon-heavy original text can confuse non-technical or junior staff — the opposite of the microlearning goal.
- **Loss of engagement / tone mismatch:** formal quotes can feel cold or intimidating in a friendly newsletter.
- **Out-of-context or outdated quoting:** lifting one line can distort meaning, and a quote may become inaccurate if the underlying policy is later revised while the snippet lingers.
- **Length / format constraints:** verbatim text is harder to fit into tight word limits or scannable bullet formats.


---
## Summary — The Prompt Development Cycle

| Stage | What we did |
|-------|-------------|
| **Design** | Built PROMPT_V1 with role, tone, format, paraphrase, and length constraints. |
| **Generate** | Produced a friendly, bulleted, sub-75-word snippet. |
| **Evaluate** | Scored output on relevance, clarity, structure, tone, length, factual accuracy. |
| **Refine** | Added strict grounding rules to block hallucinated tech/rules; created intern and quote-extraction variants. |

**Key takeaways:** explicit output constraints reliably control tone/format/length; grounding instructions ("only use the input, add nothing") are the main defense against hallucination; paraphrasing wins for clarity and engagement, while quoting wins for authority and compliance — at the cost of readability.